# 03. Khả năng mở rộng & chịu lỗi
Vẽ kết quả trong `tests/results/`:
- `sc1_sc2_*.csv`: `tests/scalability/sc1_sc2_batch_benchmark.py`
- `sc3_*.csv`: `tests/scalability/sc3_streaming_load_test.py`
- `sc4_*.csv`: `tests/scalability/sc4_es_query_load.py`
- `ft_results.csv`: `tests/fault_tolerance/chaos_test_scenarios.sh`

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RES = ROOT / "tests" / "results"

def load(pattern):
    files = sorted(RES.glob(pattern))
    return pd.concat([pd.read_csv(f).assign(file=f.name) for f in files], ignore_index=True) if files else pd.DataFrame()

bench = load("sc1_sc2_*.csv")
total = bench[bench.job == "B2-B5 total"] if len(bench) else bench

In [ ]:
sc1 = total[total.scenario == "SC1"].groupby("executors").seconds.mean() if len(total) else pd.Series(dtype=float)
if len(sc1):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sc1.plot(marker="o", ax=ax[0], title="SC1: thời gian B2-B5 theo số executor"); ax[0].set_ylabel("giây")
    (sc1.iloc[0] / sc1).plot(marker="o", ax=ax[1], title="SC1: speedup", label="đo được")
    ax[1].plot(sc1.index, sc1.index / sc1.index[0], "--", label="tuyến tính"); ax[1].legend()
else:
    print("Chưa có kết quả SC1")

In [ ]:
sc2 = total[total.scenario == "SC2"] if len(total) else total
if len(sc2):
    g = sc2.groupby("scale")[["seconds", "reviews"]].mean()
    ax = g.plot(x="reviews", y="seconds", marker="o", title="SC2: thời gian theo khối lượng dữ liệu", figsize=(7, 4))
    ax.set_xlabel("số review"); ax.set_ylabel("giây")
else:
    print("Chưa có kết quả SC2")

In [ ]:
sc3 = load("sc3_*.csv")
if len(sc3):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for label, d in sc3.groupby("label"):
        d.plot(x="target_rate", y="avg_processed_rows_per_sec", marker="o", ax=ax[0], label=label)
        d.plot(x="target_rate", y="max_offsets_behind", marker="o", ax=ax[1], label=label)
    ax[0].set_title("SC3: thông lượng xử lý"); ax[1].set_title("SC3: offset tụt hậu (consumer lag)")
else:
    print("Chưa có kết quả SC3")

In [ ]:
sc4 = load("sc4_*.csv")
if len(sc4):
    a = sc4[sc4["query"] == "ALL"]
    a.pivot_table(index="concurrency", columns="label", values="p95_ms").plot(marker="o", title="SC4: p95 độ trễ truy vấn ES (ms)")
else:
    print("Chưa có kết quả SC4")

In [ ]:
ft = RES / "ft_results.csv"
pd.read_csv(ft) if ft.exists() else "Chưa có kết quả FT"